<a href="https://colab.research.google.com/github/ravikeerthi7606/HousePricePrediction/blob/notebook/Models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%pip install -q mlflow joblib

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 56.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 34.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 34.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.7/268.7 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 228.4/228.4 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 136.5/136.5 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.7/148.7 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.

In [2]:
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
#mlflow directory
import os
from pathlib import Path

# Opt-in to filesystem tracking backend
os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"

mlflow_dir = Path(
    "/content/drive/MyDrive/ML project/HousePricePrediction/mlruns"
)

mlflow_dir.mkdir(parents=True, exist_ok=True)

#Create an experiment
import mlflow

mlflow.set_tracking_uri(mlflow_dir.as_uri())

mlflow.set_experiment("House Price Prediction")

<Experiment: artifact_location='file:///content/drive/MyDrive/ML%20project/HousePricePrediction/mlruns/769246350787289816', creation_time=1791190156868, effective_trace_archival_retention=None, experiment_id='769246350787289816', last_update_time=1791190156868, lifecycle_stage='active', name='House Price Prediction', tags={}, trace_location=None, workspace='default'>

In [4]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("shashanknecrothapa/ames-housing-dataset")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'ames-housing-dataset' dataset.
Path to dataset files: /kaggle/input/ames-housing-dataset


In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

#DON'T FORGET TO CHANGE THE PATH
df = pd.read_csv('/kaggle/input/ames-housing-dataset/AmesHousing.csv')
df.head()

,Order,PID,MS SubClass,MS Zoning,Lot Frontage,Lot Area,Street,Alley,Lot Shape,Land Contour,...,Pool Area,Pool QC,Fence,Misc Feature,Misc Val,Mo Sold,Yr Sold,Sale Type,Sale Condition,SalePrice
0,1,526301100,20,RL,141.0,31770,Pave,NaN,IR1,Lvl,...,0,NaN,NaN,NaN,0,5,2010,WD,Normal,215000
1,2,526350040,20,RH,80.0,11622,Pave,NaN,Reg,Lvl,...,0,NaN,MnPrv,NaN,0,6,2010,WD,Normal,105000
2,3,526351010,20,RL,81.0,14267,Pave,NaN,IR1,Lvl,...,0,NaN,NaN,Gar2,12500,6,2010,WD,Normal,172000
3,4,526353030,20,RL,93.0,11160,Pave,NaN,Reg,Lvl,...,0,NaN,NaN,NaN,0,4,2010,WD,Normal,244000
4,5,527105010,60,RL,74.0,13830,Pave,NaN,IR1,Lvl,...,0,NaN,MnPrv,NaN,0,3,2010,WD,Normal,189900


In [29]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression, RidgeCV, LassoCV
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, AdaBoostRegressor, GradientBoostingRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)
from sklearn.model_selection import RandomizedSearchCV
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import KFold
from sklearn.model_selection import train_test_split
import mlflow
import mlflow.sklearn
import joblib
import warnings
warnings.filterwarnings('ignore')

In [34]:
X = df.drop(columns='SalePrice')
y = df['SalePrice']

numeric_columns = X.select_dtypes(include=np.number).columns
categorical_columns = X.select_dtypes(exclude=np.number).columns


print(f'numerical columns: {len(numeric_columns)}')
print(f'categorical columns: {len(categorical_columns)}')

numerical columns: 38
categorical columns: 43


In [36]:
#imputation
numeric_pipeline_linear = Pipeline(
    steps=[('impute', SimpleImputer(strategy='median')),
           ('scale', StandardScaler())]
)

numeric_pipeline_tree = Pipeline(
    steps=[('impute', SimpleImputer(strategy='median'))]
)


categorical_pipeline = Pipeline(
    steps=[('impute', SimpleImputer(strategy='most_frequent')),
    ('ohe', OneHotEncoder(handle_unknown='ignore'))]
  )

#column transformer
preprocessor = ColumnTransformer(
    transformers=[
        ('numeric', numeric_pipeline_linear, numeric_columns),
        ('categorical', categorical_pipeline, categorical_columns)
    ])

In [9]:
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=42)

X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=42)

In [13]:
cv = KFold(n_splits=5, shuffle=True, random_state=42)

In [12]:
#Linear Regression
linear = LinearRegression()

#Pipeline
linear_pipeline = Pipeline(
    steps=[('preprocessor', preprocessor),
    ('linear regression',linear)]
)

In [26]:
#mlflow
from pathlib import Path

with mlflow.start_run(run_name='Linear Regression'):
  scores = cross_val_score(linear_pipeline, X_train, y_train, cv=cv, scoring='neg_root_mean_squared_error')
  rmse = -scores
  mlflow.log_metric('rmse_cv_mean', rmse.mean())
  mlflow.log_metric('rmse_cv_std', rmse.std())


  linear_pipeline.fit(X_train, y_train)

  rmse = np.sqrt(mean_squared_error(y_val, linear_pipeline.predict(X_val)))
  mae = mean_absolute_error(y_val, linear_pipeline.predict(X_val))
  r2 = r2_score(y_val, linear_pipeline.predict(X_val))

  # # Ensure target directory exists
  # model_path = Path("/content/drive/MyDrive/ML project/HousePricePrediction/models/GradientBoosting.joblib")
  # model_path.parent.mkdir(parents=True, exist_ok=True)

  # joblib.dump(linear_pipeline, model_path)

  # mlflow.sklearn.log_model(
  #   linear_pipeline,
  #   "Gradient boost Regressor",
  #   serialization_format='cloudpickle'
  # )

  mlflow.log_metric('rmse', rmse)
  mlflow.log_metric('mae', mae)
  mlflow.log_metric('r2', r2)

In [21]:
from sklearn.linear_model import Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
import numpy as np
import mlflow

# Define the candidate models
models = {
    'Ridge': Ridge(alpha=1.0),
    'Lasso': Lasso(alpha=1.0, max_iter=2000),
    'Random Forest Regressor': RandomForestRegressor(n_estimators=100, random_state=42),
    'Gradient Boosting Regressor': GradientBoostingRegressor(n_estimators=100, random_state=42)
}

# Iterate and log metrics for each model dynamically
for model_name, model in models.items():
    print(f"Running experiment for: {model_name}...")

    # 1. Create pipeline with the current model
    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('regressor', model)
    ])

    # 2. Start MLflow run
    with mlflow.start_run(run_name=model_name):
        # Log model parameters if desired (e.g., regressor class name)
        mlflow.log_param('model_class', model.__class__.__name__)

        # 3. Cross-Validation on training set
        scores = cross_val_score(
            pipeline, X_train, y_train, cv=cv, scoring='neg_root_mean_squared_error'
        )
        rmse_cv = -scores
        mlflow.log_metric('rmse_cv_mean', rmse_cv.mean())
        mlflow.log_metric('rmse_cv_std', rmse_cv.std())

        # 4. Fit on entire training set and evaluate on validation set
        pipeline.fit(X_train, y_train)
        val_predictions = pipeline.predict(X_val)

        val_rmse = np.sqrt(mean_squared_error(y_val, val_predictions))
        val_mae = mean_absolute_error(y_val, val_predictions)
        val_r2 = r2_score(y_val, val_predictions)

        # 5. Log evaluation metrics
        mlflow.log_metric('rmse', val_rmse)
        mlflow.log_metric('mae', val_mae)
        mlflow.log_metric('r2', val_r2)

        print(f"Finished {model_name}. Validation R2: {val_r2:.4f}, RMSE: {val_rmse:.2f}\n")

Running experiment for: Ridge...
Finished Ridge. Validation R2: 0.0627, RMSE: 80114.50

Running experiment for: Lasso...
Finished Lasso. Validation R2: 0.9400, RMSE: 20275.46

Running experiment for: Random Forest Regressor...
Finished Random Forest Regressor. Validation R2: 0.9277, RMSE: 22243.46

Running experiment for: Gradient Boosting Regressor...
Finished Gradient Boosting Regressor. Validation R2: 0.9361, RMSE: 20917.29



In [28]:
# Fetch and compare all runs from our active MLflow experiment
experiment = mlflow.get_experiment_by_name("House Price Prediction")
runs = mlflow.search_runs(experiment_ids=[experiment.experiment_id])

# Sort and display by Validation RMSE (ascending is better)
columns_to_show = [
    'tags.mlflow.runName',
    'metrics.rmse',
    'metrics.mae',
    'metrics.r2',
    'metrics.rmse_cv_mean',
    'metrics.rmse_cv_std',
    'params.model_class'
]

# Keep only columns that exist in the DataFrame to prevent errors
available_cols = [col for col in columns_to_show if col in runs.columns]
display(runs[available_cols].sort_values('metrics.rmse', ascending=True))

,tags.mlflow.runName,metrics.rmse,metrics.mae,metrics.r2,metrics.rmse_cv_mean,metrics.rmse_cv_std,params.model_class
3,Lasso,20275.461716,14322.419929,0.939967,30375.377903,6986.044930,Lasso
1,Gradient Boosting Regressor,20917.288805,14226.756079,0.936106,24877.806492,2941.049902,GradientBoostingRegressor
2,Random Forest Regressor,22243.463437,14589.070410,0.927747,27537.379239,3697.454770,RandomForestRegressor
0,Linear Regression,39779.735651,27772.628885,0.768913,46683.368500,5119.655251,None
5,Linear Regression,39779.735651,27772.628885,0.768913,NaN,NaN,None
6,Linear Regression,47497.377897,30632.089413,0.687025,NaN,NaN,None
4,Ridge,80114.495747,60023.169447,0.062712,75713.670333,2638.685236,Ridge
7,Linear Regression,NaN,NaN,NaN,NaN,NaN,None
8,Gradient Boosting Regressor,NaN,14683.750624,0.911283,NaN,NaN,None
9,Adaboost Regressor,NaN,25109.502708,0.837025,NaN,NaN,None


In [25]:
runs

,run_id,experiment_id,status,artifact_uri,start_time,end_time,metrics.rmse_cv_mean,metrics.r2,metrics.rmse_cv_std,metrics.mae,...,params.Trees__monotonic_cst,params.Trees__bootstrap,params.Trees__oob_score,params.Trees__max_samples,tags.mlflow.user,tags.mlflow.runName,tags.mlflow.source.type,tags.mlflow.source.name,tags.estimator_class,tags.estimator_name
0,7868594b9d79406ba4dfeb9df4647c89,769246350787289816,FINISHED,file:///content/drive/MyDrive/ML%20project/Hou...,2026-10-06 16:59:35.748000+00:00,2026-10-06 16:59:49.359000+00:00,24877.806492,0.936106,2941.049902,14226.756079,...,None,None,None,None,root,Gradient Boosting Regressor,NOTEBOOK,fileId=14kcRMAGna1d66txEO4_RjRtLeS5ySY86,None,None
1,68d52343040046238e4750516937d261,769246350787289816,FINISHED,file:///content/drive/MyDrive/ML%20project/Hou...,2026-10-06 16:57:09.388000+00:00,2026-10-06 16:59:35.732000+00:00,27537.379239,0.927747,3697.454770,14589.070410,...,None,None,None,None,root,Random Forest Regressor,NOTEBOOK,fileId=14kcRMAGna1d66txEO4_RjRtLeS5ySY86,None,None
2,825550cd28e54a6bb453381939b1ed76,769246350787289816,FINISHED,file:///content/drive/MyDrive/ML%20project/Hou...,2026-10-06 16:56:40.565000+00:00,2026-10-06 16:57:09.371000+00:00,30375.377903,0.939967,6986.044930,14322.419929,...,None,None,None,None,root,Lasso,NOTEBOOK,fileId=14kcRMAGna1d66txEO4_RjRtLeS5ySY86,None,None
3,b7c8cabaf4804be9b0d98a7e7cdb5757,769246350787289816,FINISHED,file:///content/drive/MyDrive/ML%20project/Hou...,2026-10-06 16:56:39.739000+00:00,2026-10-06 16:56:40.549000+00:00,75713.670333,0.062712,2638.685236,60023.169447,...,None,None,None,None,root,Ridge,NOTEBOOK,fileId=14kcRMAGna1d66txEO4_RjRtLeS5ySY86,None,None
4,4fd8a1a827c849b08d9503277aa0ff9a,769246350787289816,FINISHED,file:///content/drive/MyDrive/ML%20project/Hou...,2026-10-06 16:50:59.605000+00:00,2026-10-06 16:51:01.258000+00:00,NaN,0.768913,NaN,27772.628885,...,None,None,None,None,root,Linear Regression,NOTEBOOK,fileId=14kcRMAGna1d66txEO4_RjRtLeS5ySY86,None,None
5,acd7bded84524ebf873f119782255888,769246350787289816,FINISHED,file:///content/drive/MyDrive/ML%20project/Hou...,2026-10-06 16:49:53.772000+00:00,2026-10-06 16:49:55.054000+00:00,NaN,0.687025,NaN,30632.089413,...,None,None,None,None,root,Linear Regression,NOTEBOOK,fileId=14kcRMAGna1d66txEO4_RjRtLeS5ySY86,None,None
6,6ec0d42cde7345d89971adb0cd1d2140,769246350787289816,FAILED,file:///content/drive/MyDrive/ML%20project/Hou...,2026-10-06 16:49:39.637000+00:00,2026-10-06 16:49:39.845000+00:00,NaN,NaN,NaN,NaN,...,None,None,None,None,root,Linear Regression,NOTEBOOK,fileId=14kcRMAGna1d66txEO4_RjRtLeS5ySY86,None,None
7,3c00cce4184f4eb9ba92f2d3db71cd16,769246350787289816,FINISHED,file:///content/drive/MyDrive/ML%20project/Hou...,2026-10-05 19:05:23.083000+00:00,2026-10-05 19:05:37.654000+00:00,NaN,0.911283,NaN,14683.750624,...,None,None,None,None,root,Gradient Boosting Regressor,NOTEBOOK,fileId=14kcRMAGna1d66txEO4_RjRtLeS5ySY86,sklearn.pipeline.Pipeline,Pipeline
8,1f8ba719bb174071b518b4de6e1592cd,769246350787289816,FINISHED,file:///content/drive/MyDrive/ML%20project/Hou...,2026-10-05 18:58:00.523000+00:00,2026-10-05 18:58:14.599000+00:00,NaN,0.837025,NaN,25109.502708,...,None,None,None,None,root,Adaboost Regressor,NOTEBOOK,fileId=14kcRMAGna1d66txEO4_RjRtLeS5ySY86,sklearn.pipeline.Pipeline,Pipeline
9,7be9bfca4f5948d19ac9a13c26ccabc1,769246350787289816,FINISHED,file:///content/drive/MyDrive/ML%20project/Hou...,2026-10-05 18:54:28.227000+00:00,2026-10-05 18:54:44.790000+00:00,NaN,0.860686,NaN,19655.193293,...,None,True,False,None,root,Random Tree Regressor,NOTEBOOK,fileId=14kcRMAGna1d66txEO4_RjRtLeS5ySY86,sklearn.pipeline.Pipeline,Pipeline


I've done big mistakes while training models using for loop. I forgot to use scaling in ridge, lasso and linear regression that's why scores are pretty low. Though I didn't scale the data,How did lasso model perform exceptionally well than other tree based models

In [32]:
# Randomized Search CV Parameter Distributions

lasso = Lasso()
rf = RandomForestRegressor()
gb = GradientBoostingRegressor()

params_distribution_lasso = {
    'alpha': [0.001, 0.01, 0.1, 1.0, 10.0],
    'max_iter': [100, 500, 1000, 2000],
    'tol': [1e-3, 1e-4, 1e-5]
}

params_distribution_rf = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2', None]
}

params_distribution_gb = {
    'n_estimators': [50, 100, 200],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'max_depth': [3, 4, 5, 6],
    'min_samples_split': [2, 5, 10],
    'subsample': [0.8, 0.9, 1.0]
}

model_dict = {
    'Lasso': (lasso, params_distribution_lasso),
    'Random Forest': (rf, params_distribution_rf),
    'Gradient Boosting': (gb, params_distribution_gb)
}

In [37]:
for model_name, model_info in model_dict.items():
    print(f"Running RandomizedSearchCV for: {model_name}...")

    # 1. Create pipeline with the current model
    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('regressor', model_info[0])
    ])

    # 2. Start MLflow run
    with mlflow.start_run(run_name=f"{model_name} Tuned"):

        # Use parameter grid namespaced for the 'regressor' step in the pipeline
        param_dist = {f'regressor__{k}': v for k, v in model_info[1].items()}

        RandomCV = RandomizedSearchCV(
            estimator=pipeline,
            param_distributions=param_dist,
            n_iter=10,
            cv=cv,
            scoring='neg_root_mean_squared_error',
            random_state=42
        )
        RandomCV.fit(X_train, y_train)

        # Log best parameters and training score
        mlflow.log_params(RandomCV.best_params_)
        mlflow.log_metric('rmse_cv_best', -RandomCV.best_score_)

        # Use best_estimator_ instead of best_model_
        best_pipeline = RandomCV.best_estimator_
        val_predictions = best_pipeline.predict(X_val)

        val_rmse = np.sqrt(mean_squared_error(y_val, val_predictions))
        val_mae = mean_absolute_error(y_val, val_predictions)
        val_r2 = r2_score(y_val, val_predictions)

        # Log validation metrics
        mlflow.log_metric('rmse', val_rmse)
        mlflow.log_metric('mae', val_mae)
        mlflow.log_metric('r2', val_r2)

        print(f"Finished {model_name}. Validation R2: {val_r2:.4f}, RMSE: {val_rmse:.2f}\n")

Running RandomizedSearchCV for: Lasso...


KeyboardInterrupt: 

In [38]:
for model_name, model_info in model_dict.items():
    print(f"Running RandomizedSearchCV for: {model_name}...")

    # 1. Dynamically select the correct numerical pipeline
    if model_name == 'Lasso':
        current_numeric_pipeline = numeric_pipeline_linear
    else:
        current_numeric_pipeline = numeric_pipeline_tree

    current_preprocessor = ColumnTransformer(
        transformers=[
            ('numeric', current_numeric_pipeline, numeric_columns),
            ('categorical', categorical_pipeline, categorical_columns)
        ])

    # 2. Create pipeline with the selected preprocessor
    pipeline = Pipeline(steps=[
        ('preprocessor', current_preprocessor),
        ('regressor', model_info[0])
    ])

    # 3. Start MLflow run
    with mlflow.start_run(run_name=f"{model_name} Tuned"):

        # Namespace parameter grids
        param_dist = {f'regressor__{k}': v for k, v in model_info[1].items()}

        RandomCV = RandomizedSearchCV(
            estimator=pipeline,
            param_distributions=param_dist,
            n_iter=10,
            cv=cv,
            scoring='neg_root_mean_squared_error',
            random_state=42,
            n_jobs=-1  # Utilize all available CPU cores for faster tuning
        )
        RandomCV.fit(X_train, y_train)

        # Log best parameters and training score
        mlflow.log_params(RandomCV.best_params_)
        mlflow.log_metric('rmse_cv_best', -RandomCV.best_score_)

        # Predict and evaluate
        best_pipeline = RandomCV.best_estimator_
        val_predictions = best_pipeline.predict(X_val)

        val_rmse = np.sqrt(mean_squared_error(y_val, val_predictions))
        val_mae = mean_absolute_error(y_val, val_predictions)
        val_r2 = r2_score(y_val, val_predictions)

        # Log validation metrics
        mlflow.log_metric('rmse', val_rmse)
        mlflow.log_metric('mae', val_mae)
        mlflow.log_metric('r2', val_r2)

        print(f"Finished {model_name}. Validation R2: {val_r2:.4f}, RMSE: {val_rmse:.2f}\n")

Running RandomizedSearchCV for: Lasso...
Finished Lasso. Validation R2: 0.9410, RMSE: 20098.33

Running RandomizedSearchCV for: Random Forest...
Finished Random Forest. Validation R2: 0.9227, RMSE: 23000.22

Running RandomizedSearchCV for: Gradient Boosting...
Finished Gradient Boosting. Validation R2: 0.9395, RMSE: 20348.97

